In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tqdm import tqdm
from tensorflow.keras.callbacks import Callback

In [ ]:
# Read data from text files
def read_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    return content


In [ ]:
# Set the path to your dataset directory
dataset_path = "/kaggle/input/typology/Typology"

In [ ]:
# List all the files in the dataset directory
all_files = os.listdir(dataset_path)

In [ ]:
# Create empty lists to store content and labels
content_list = []
typology_list = []

In [ ]:
# Iterate through all files in the dataset directory
for file_name in tqdm(all_files, desc="Reading files"):
    file_path = os.path.join(dataset_path, file_name)
    
    # Check if the path is a file (not a subdirectory)
    if os.path.isfile(file_path):
        # Read content and label from the file
        content = read_data(file_path)
        typology_start = content.find("Political Typology:")
        typology = content[typology_start + len("Political Typology:"):].strip()

        content_list.append(content)
        typology_list.append(typology)

In [ ]:
# Create a DataFrame
df = pd.DataFrame({'Content': content_list, 'Typology': typology_list})

In [ ]:
# Split the data into training and testing sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
# Assuming train_df is your training DataFrame
MAX_SEQUENCE_LENGTH = 1500
print("Maximum Sequence Length:", MAX_SEQUENCE_LENGTH)


In [ ]:
# Tokenize and pad sequences
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_df['Content'])
X_train = pad_sequences(tokenizer.texts_to_sequences(train_df['Content']), maxlen=MAX_SEQUENCE_LENGTH)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_df['Content']), maxlen=MAX_SEQUENCE_LENGTH)

In [ ]:
# Binarize the labels
mlb = MultiLabelBinarizer()
y_train = mlb.fit_transform(train_df['Typology'])
y_test = mlb.transform(test_df['Typology'])

In [ ]:
# Assuming you have already fit the tokenizer on your training data
vocab_size = len(tokenizer.word_index) + 1  # Add 1 for the reserved index 0

embedding_dim = 100

In [ ]:
# Build the model
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=MAX_SEQUENCE_LENGTH))
model.add(LSTM(100))
model.add(Dense(len(mlb.classes_), activation='sigmoid'))  # Sigmoid for multi-label classification

In [ ]:
# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:


# Define a custom callback for tqdm progress bar
class TQDMProgressBar(Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self.epochs = self.params['epochs']
        self.epoch_progress_bar = tqdm(total=self.epochs, desc="Epoch Progress", position=0)

    def on_epoch_end(self, epoch, logs=None):
        self.epoch_progress_bar.update(1)
        self.epoch_progress_bar.close()

    def on_batch_begin(self, batch, logs=None):
        self.epoch_progress_bar.set_postfix(Batch=batch, refresh=False)
        self.epoch_progress_bar.update(1)

# Use the custom callback during model training
tqdm_callback = TQDMProgressBar()


In [ ]:
# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')